# METAFlux lung tumours

This notebook runs the analysis on fluxes from a second method, METAFlux ([Huang et al., Nat Commun 2023](https://doi.org/10.1038/s41467-023-40457-w)), which predicts each sample's reaction fluxes from its gene expression. The dataset holds the fluxes METAFlux predicts for 1,017 primary lung tumours of The Cancer Genome Atlas, one per patient: 516 adenocarcinomas (LUAD) and 501 squamous cell carcinomas (LUSC). They were computed with METAFlux's bulk pipeline and the human blood medium the paper used, from each tumour's log2(TPM + 1) expression read through UCSC Xena. Comparing these two tumour types, Huang et al. report that "LUSC tumors had a higher GUF than did LUAD tumors", GUF being the glucose uptake flux (Fig. 3a, two-sided Wilcoxon rank-sum test). This notebook makes the same comparison and reads that reaction, as the server's METAFlux tutorial does. What notebook 01 introduced is used here without introduction.

## Before you run it

Install the repository with its development dependencies, then install the example data:

```bash
python data/download_datasets.py
```

That script fetches the same author-maintained archive the Metabis server deploys, so the notebook and the application read byte-identical inputs.

In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from sklearn.metrics import silhouette_score

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings(
    "ignore",
    message="n_jobs value 1 overridden to 1 by setting random_state.*",
)


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    AnalysisWorkspace,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    GroupComparisonSpec,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    build_reduction_presentation,
    element_details,
    plot_reduction_artifact,
    plot_statistical_scatterplots,
    plot_transport_view,
)
from metabis.analysis.dimensionality_reduction import DimensionalityReduction
from metabis.analysis_workspace import RevisionKind
from metabis.domain.expression_matrix import DenseExpressionMatrix
from metabis.version import __version__ as metabis_version
from metabis.viz.transport_view import transport_rows

output_notebook()

In [ ]:
repo_root = find_metabis_repo_root_for_import(Path.cwd().resolve())
data_dir = repo_root / "data"
required_example_files = (
    "metaflux_tcga_lung_reactions.tsv",
    "metaflux_tcga_lung_expression.tsv",
    "metaflux_tcga_lung_metadata.tsv",
)
missing = [name for name in required_example_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing example files: "
        + ", ".join(missing)
        + ". Run `python data/download_datasets.py`."
    )

reaction_data_path, expression_data_path, metadata_path = (
    data_dir / name for name in required_example_files
)

print(f"Metabis version: {metabis_version}")
for path in (reaction_data_path, expression_data_path, metadata_path):
    print(f"Using {path.name}")

## 1. Input data

METAFlux gives each reaction one signed flux, negative when the reaction runs backwards. Read in single direction (`data_mode="single"`), Metabis splits a reversible reaction into a forward direction, the positive part of its flux, and a reverse one (`_back`), the size of the negative part, and tests each. The fluxes are compared as they are, on their linear scale: the preprocessing applies no transform.

The reactions are named by Human1's older HMR identifiers (`rxnHMR2ID`), which Metabis reads. Human1 is the model METAFlux ran on and the one Metabis analyses with, so no reaction is realigned or turned round.

`zero_below=1e-6` reads values smaller than 1e-6 in magnitude as zero. A solver stops within a tolerance of the exact answer, so a flux that should be zero comes out as a tiny value of either sign.

In [ ]:
model = MetabolicModel(use_cache=True)
reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="single",
    metabolic_model=model,
    reaction_name_format="rxnHMR2ID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(zero_below=1e-6),
)
assert reaction_validation["status"] == "success", reaction_validation["messages"]

alignment = reaction_validation["alignment_report"]
zero_notice = reaction_validation["warnings"][0]
n_samples, n_directions = reaction_data.get_reaction_df().shape

print(zero_notice)
print(f"Samples: {n_samples:,}")
print(f"Reaction directions: {n_directions:,}")
print(
    f"Human1 coverage: {alignment['model_coverage_percent']:.1f}% "
    f"({alignment['base_reactions_mapped']:,} of {alignment['total_model_reactions']:,} reactions)"
)
print(
    f"Subsystems: {alignment['subsystems_with_reactions']} of "
    f"{alignment['total_subsystems']} covered"
)
print(f"Not mapped: {alignment['unmapped_count']:,} reactions")
print(f"Scale for the log2 fold change: {reaction_data.fold_change_scale}")

assert zero_notice.startswith("7,805,763 of the reactions' values (62.8%)")
assert (n_samples, n_directions) == (1017, 20799)
assert f"{alignment['model_coverage_percent']:.1f}" == "95.9"
assert (alignment["subsystems_with_reactions"], alignment["total_subsystems"]) == (146, 154)
assert alignment["unmapped_count"] == 749
assert reaction_data.fold_change_scale == "linear"

7,805,763 values, 62.8% of all the values, are read as zero. The 1,017 tumours keep 20,799 reaction directions, which cover 95.9% of Human1 and 146 of its 154 subsystems; 749 reactions have no Human1 counterpart. The fluxes are on a linear scale, so a reaction direction's log2 fold change is the log2 of the ratio of the group means.

Read as zero, the small values leave 20,799 directions to test instead of the 23,996 the file gives with the threshold at 0:

In [ ]:
kept_reaction_data, kept_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="single",
    metabolic_model=model,
    reaction_name_format="rxnHMR2ID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(zero_below=0.0),
)
assert kept_validation["status"] == "success", kept_validation["messages"]
kept_directions = kept_reaction_data.get_reaction_df().shape[1]
print(f"Reaction directions with the threshold at 0: {kept_directions:,}")
assert kept_directions == 23996

The expression METAFlux was given, log2(TPM + 1), is used as it is. `ExpressionPreprocessingConfig(input_scale="log2")` applies nothing and states the scale, so a gene's log2 fold change is the difference of the group means.

In [ ]:
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(input_scale="log2"),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
assert expression_validation["status"] == "success", expression_validation["messages"]

expression_alignment = expression_validation["alignment_report"]
print(
    f"Metabolic genes: {expression_alignment['mapped_count']:,}, "
    f"{expression_alignment['model_coverage_percent']:.1f}% of Human1's "
    f"{expression_alignment['total_model_genes']:,}"
)
print(f"Scale for the log2 fold change: {expression_data.fold_change_scale}")

assert expression_alignment["mapped_count"] == 2792
assert f"{expression_alignment['model_coverage_percent']:.1f}" == "97.3"
assert expression_data.fold_change_scale == "log2"

The expression holds 2,792 metabolic genes, 97.3% of Human1's. The sample sheet is aligned to the reaction data by its `sample` column, as in notebook 01:

In [ ]:
metadata = SampleMetadata.from_table(metadata_path, sample_id_column="sample")
dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
metadata_alignment = dm.update_sample_metadata(metadata)

print(f"Aligned samples: {len(metadata_alignment.aligned_samples):,}")
print(f"Columns: {list(metadata.columns)}")

assert len(metadata_alignment.aligned_samples) == 1017
assert list(metadata.columns) == ["patient", "project", "stage", "sex", "age"]

All 1,017 tumours are aligned. Besides `project`, the sheet holds each tumour's `patient`, `stage`, `sex` and `age`.

## 2. Groups and the reaction map

The two groups, one per tumour type, come from the TCGA project each tumour belongs to. Their summaries are computed and installed with an `ExperimentAnalysis`, as in notebook 01.

In [ ]:
dm.load_groups_from_metadata(["project"])
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
group_analysis = ExperimentAnalysis(dm, auto_run=False)
outcome = group_analysis.run_analysis(include_pca=False)
assert outcome["status"] == "success", outcome["problems"]
dm.install_experiment_analysis(analysis_snapshot.provenance, group_analysis)
dm.refresh_experiment_analysis_presentation()

group_sizes = {group: len(dm.get_group_samples(group)) for group in sorted(dm.get_group_ids())}
tumour_type = pd.Series(
    {sample: group for group in group_sizes for sample in dm.get_group_samples(group)}
)
print(group_sizes)
assert group_sizes == {"LUAD": 516, "LUSC": 501}

LUAD holds 516 tumours and LUSC 501.

Fluxes span orders of magnitude: read as loaded, the few reactions with the largest fluxes would decide the map. So the reactions are prepared for the map as Huang et al. prepared their map of these tumours (Fig. 3b and its Supplementary Methods): a signed cube root, the 3,000 reactions of highest standardized variance, as Seurat chooses them, and each reaction scaled to mean 0 and standard deviation 1, so every reaction weighs the same. The server fits the map with the `DimensionalityReduction` calculator, which takes these settings, on the reaction frame; the notebook makes the same call.

In [ ]:
map_preparation = {
    "feature_transform": "signed_cuberoot",
    "feature_selection": "standardized_variance",
    "scale_features": True,
}
reduction_snapshot = dm.capture_analysis_snapshot(
    {RevisionKind.DATASET, RevisionKind.PREPROCESSING}
)
reaction_matrix = DenseExpressionMatrix(reaction_data.get_reaction_df())
reducer = DimensionalityReduction()


def fit_map(method, **preparation):
    calculate = reducer.calculate_pca if method == "PCA" else reducer.calculate_umap
    outcome = calculate(
        matrix=reaction_matrix,
        data_type="reactions",
        data_source_name="reaction data",
        n_variable_features=3000,
        **preparation,
    )
    assert outcome["status"] == "success", outcome["messages"]
    return outcome["result"].build_artifact(
        provenance=reduction_snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    )


def silhouette(artifact):
    return silhouette_score(
        artifact.coordinates[:, :2], tumour_type[list(artifact.sample_ids)]
    )


pca_map = fit_map("PCA", **map_preparation)
pca_plot = plot_reduction_artifact(
    pca_map, build_reduction_presentation(dm, pca_map, width=650, height=480)
)
show(pca_plot)

In [ ]:
pc1 = pd.Series(pca_map.coordinates[:, 0], index=list(pca_map.sample_ids))
lusc_side = np.sign(pc1[tumour_type == "LUSC"].median())
on_lusc_side = (np.sign(pc1) == lusc_side).groupby(tumour_type).mean()
separation_note = next(
    item.text for item in pca_plot.below if "silhouette" in str(getattr(item, "text", ""))
)
prepared_silhouette = silhouette(pca_map)
as_loaded_silhouette = silhouette(fit_map("PCA"))

print(f"Reactions on the map: {pca_map.n_variable_features_used:,}")
print(f"Share of each type on the LUSC side of PC1: {on_lusc_side.round(2).to_dict()}")
print(f"The note under the map: {separation_note}")
print(f"Silhouette, reactions prepared: {prepared_silhouette:.2f}")
print(f"Silhouette, reactions as loaded: {as_loaded_silhouette:.2f}")

assert pca_map.n_variable_features_used == 3000
assert on_lusc_side["LUSC"] > 0.5 > on_lusc_side["LUAD"]
assert "is poor" in separation_note and "silhouette 0.21" in separation_note
assert f"{prepared_silhouette:.2f}" == "0.21"
assert f"{as_loaded_silhouette:.2f}" == "0.01"

On the PCA, LUSC tumours lie mostly to one side and LUAD tumours to the other, overlapping in the middle. The note under the map calls the separation of the groups poor, with a silhouette of 0.21, against 0.01 with the reactions read as loaded.

With UMAP as the method, the tumours fall into two islands, each holding both types, LUSC on one side of each and LUAD on the other:

In [ ]:
umap_map = fit_map("UMAP", **map_preparation)
show(
    plot_reduction_artifact(
        umap_map, build_reduction_presentation(dm, umap_map, width=650, height=480)
    )
)

The islands come from the fluxes METAFlux computed, in which a few reactions run one way in part of the tumours and the other way in the rest; they do not follow the tumour types.

The preparation changes the map only: section 3 tests the fluxes as loaded.

## 3. LUSC against LUAD

Every reaction direction and every gene is tested between the two tumour types. LUSC is the Target, so every signed result is LUSC against LUAD: positive where LUSC is higher. The test is the Wilcoxon rank-sum test, the one Huang et al. compared glucose uptake with, corrected with Benjamini-Hochberg. The comparison is installed in the workspace and judged at adjusted p below 0.05, as in notebook 01.

In [ ]:
comparison = GroupComparison(
    spec=GroupComparisonSpec.pairwise(
        ("LUSC",), ("LUAD",), target_label="LUSC", baseline_label="LUAD"
    ),
    test="wilcoxon",
    correction="fdr_bh",
)
comparison.run_test(dm, for_genes=False)
comparison.run_test(dm, for_genes=True)
comparison_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
comparison_artifact = comparison.build_artifact(
    provenance=comparison_snapshot.provenance,
    source_fingerprints=dict(dm.source_fingerprints),
)
dm.install_group_comparison_artifact(comparison_snapshot.provenance, comparison_artifact)
dm.refresh_group_comparison_presentation()


def judge(cohens_d=0.0):
    counts = {}
    for family, for_genes in (("reaction directions", False), ("genes", True)):
        significant, _info = dm.apply_comparison_thresholds(
            pval_threshold=0.05,
            cohens_d_threshold=cohens_d,
            log2_fc_threshold=0.0,
            mean_difference_threshold=0.0,
            for_genes=for_genes,
        )
        counts[family] = len(significant)
    return counts


significant_counts = judge()
print(significant_counts)
assert significant_counts == {"reaction directions": 4897, "genes": 2339}

4,897 reaction directions and 2,339 genes differ at adjusted p below 0.05.

METAFlux's fluxes are on a linear scale, so each reaction direction gets a log2 fold change: the log2 of the ratio of the two groups' mean fluxes. The volcano plot sets each reaction direction by its log2 fold change and its adjusted p:

In [ ]:
reaction_results = dm.get_group_comparison_artifact().get_test_results(for_genes=False)
both_flowing = (reaction_results["group_1_mean"] > 0) & (reaction_results["group_2_mean"] > 0)
ratio = np.log2(
    reaction_results.loc[both_flowing, "group_1_mean"]
    / reaction_results.loc[both_flowing, "group_2_mean"]
)
assert np.allclose(reaction_results.loc[both_flowing, "log2_fc"], ratio)

show(
    plot_statistical_scatterplots(
        dm,
        return_plot=True,
        show_thresholds=True,
        for_genes=False,
        volcano="log2_fc",
        volcano_y="adjusted",
        sides=("LUAD", "LUSC"),
    )
)

Far out at the sides are directions whose flux is near zero in one group: a ratio to a tiny mean is large, however small the fluxes. The five significant directions farthest out have a mean of zero in one group:

In [ ]:
significant_reactions = reaction_results[reaction_results["is_significant"]]
farthest = significant_reactions.loc[
    significant_reactions["log2_fc"].abs().nlargest(5).index,
    ["log2_fc", "cohens_d", "group_1_mean", "group_2_mean", "adjusted_pval"],
].rename(columns={"group_1_mean": "LUSC mean", "group_2_mean": "LUAD mean"})
assert (farthest[["LUSC mean", "LUAD mean"]].min(axis=1) == 0).all()
farthest

### Keep the larger differences

With about 500 tumours a side, even very small differences test significant. A smallest |Cohen's d| asks a feature to reach an effect size as well as the p-value: 0.2, 0.5 and 0.8 are Cohen's small, medium and large effects. Set at 0.5, medium:

In [ ]:
print(
    "Smallest |Cohen's d| among the significant directions: "
    f"{significant_reactions['cohens_d'].abs().min():.3f}"
)
significant_counts = judge(cohens_d=0.5)
print(significant_counts)
assert significant_counts == {"reaction directions": 1018, "genes": 1081}

1,018 reaction directions and 1,081 genes pass both thresholds. Section 4 reads this analysis.

### The values read as zero

Kept, the values smaller than 1e-6 would be tested as well. With the threshold at 0, 7,233 reaction directions test significant, and in 3,419 of them both groups' mean flux is below 1e-6:

In [ ]:
kept_dm = AnalysisWorkspace(metabolic_model=model, reaction_data=kept_reaction_data)
kept_dm.update_sample_metadata(metadata)
kept_dm.load_groups_from_metadata(["project"])
kept_comparison = GroupComparison(
    spec=GroupComparisonSpec.pairwise(
        ("LUSC",), ("LUAD",), target_label="LUSC", baseline_label="LUAD"
    ),
    test="wilcoxon",
    correction="fdr_bh",
)
kept_comparison.run_test(kept_dm, for_genes=False)
kept_significant, _info = kept_comparison.define_significant_elements(
    pval_threshold=0.05,
    cohens_d_threshold=0.0,
    log2_fc_threshold=0.0,
    mean_difference_threshold=0.0,
    for_genes=False,
)
kept_results = kept_comparison.get_test_results(for_genes=False).loc[list(kept_significant)]
below = (kept_results["group_1_mean"] < 1e-6) & (kept_results["group_2_mean"] < 1e-6)
print(f"Significant reaction directions: {len(kept_significant):,}")
print(f"Both mean fluxes below 1e-6: {below.sum():,}")
assert (len(kept_significant), below.sum()) == (7233, 3419)

## 4. Glucose uptake

Glucose uptake is an exchange reaction: Human1's MAR09034, Exchange of glucose, through which the model takes extracellular glucose in or releases it. METAFlux writes uptake as a negative flux, so Metabis reads it as the reaction's reverse direction:

In [ ]:
glucose = "MAR09034"
glucose_fluxes = reaction_data.get_reaction_df()[[glucose, f"{glucose}_back"]]
print((glucose_fluxes > 0).sum().rename("tumours with flux in that direction"))
assert (glucose_fluxes[glucose] == 0).all() and (glucose_fluxes[f"{glucose}_back"] > 0).all()

The uptake, the reverse direction, carries flux in all 1,017 tumours, and the release none.

The Exchange/demand reactions subsystem is drawn as rows, one per metabolite, since each exchange moves one metabolite. The 20 that change most have rows of their own, and the rest share the last. Each row gives the metabolite's Delta score, and each of its reactions as a dot at its Cohen's d, from the analysis in view, the one with the effect-size threshold. `plot_transport_view` draws it with the Delta score's default settings, as the server does.

In [ ]:
exchange = next(
    subsystem
    for subsystem in model.get_subsystems()
    if model.get_subsystem_name(subsystem) == "Exchange/demand reactions"
)
delta_options = {
    "metric": "cohens_d",
    "transform": "sign",
    "numerator_filter": "significant",
    "denominator_filter": "all",
    "reaction_direction_criterion": dm.reaction_direction_criterion.value,
}
dm.update_subsystem_delta_config(delta_options)
show(
    plot_transport_view(
        dm,
        exchange,
        metric="cohens_d",
        delta_options=delta_options,
        hover_detail="medium",
    )
)

In [ ]:
installed = dm.get_group_comparison_artifact()
installed_results = installed.get_test_results(for_genes=False)
rows = [
    row.name
    for row in transport_rows(
        model,
        exchange,
        measured=installed_results.index,
        significant=installed.get_significant_elements(for_genes=False),
        strength=installed_results["cohens_d"].to_dict(),
    )
]
print(rows)
assert len(rows) == 21 and rows[-1].startswith("Other metabolites")
assert "glucose" in rows[:20]

Glucose has a row of its own. A click on its dot opens the reaction's details beside the rows; `element_details` returns them:

In [ ]:
details = element_details(dm, "reaction", glucose)
statistics = details["statistics"]
means = {
    violin["direction"]: {group["name"]: group["mean"] for group in violin["groups"]}
    for violin in statistics["distributions"]
}
directions = sorted(statistics["directions"], key=lambda direction: not direction["main"])
print(f"{details['id']}, {details['name']}: {details['subsystem']}")
print(f"Read in the direction more active in {statistics['more_active_in']}")
for direction in directions:
    print(
        f"{direction['direction']}: significant {direction['significant']}, "
        f"Cohen's d {direction['cohens_d']:.2f}, log2 fold change {direction['log2_fc']:.2f}, "
        f"adjusted p {direction['adjusted_pval']:.1e}, "
        f"mean flux LUAD {means[direction['direction']]['LUAD']:.5f}, "
        f"LUSC {means[direction['direction']]['LUSC']:.5f}"
    )
uptake, release = directions
uptake_ratio = means["backward"]["LUSC"] / means["backward"]["LUAD"]
scores_counted = {spread["count"] for spread in uptake["spreads"]}
print(f"LUSC's mean uptake over LUAD's: {uptake_ratio:.2f}")
print(f"Reaction scores each is set among: {scores_counted}")

assert details["name"] == "Exchange of glucose"
assert statistics["more_active_in"] == "LUAD"
assert uptake["direction"] == "backward" and uptake["significant"]
assert release["direction"] == "forward" and not release["significant"]
assert f"{uptake['cohens_d']:.2f}" == "1.04"
assert f"{uptake['log2_fc']:.2f}" == "0.44"
assert f"{uptake_ratio:.2f}" == "1.36"
assert f"{uptake['adjusted_pval']:.1e}" == "1.1e-49"
assert scores_counted == {20799}

Glucose uptake is higher in LUSC: Cohen's d 1.04, a log2 fold change of 0.44 (LUSC's mean uptake 1.36 times LUAD's), adjusted p 1.1 × 10<sup>−49</sup>, each set among the comparison's 20,799 reaction scores. This is the finding Huang et al. report, with the same test (Fig. 3a). The details read a reversible reaction in the direction more active in the baseline group, LUAD, which here is the uptake; the release is not significant.